# **Project Name** — Amazon Diwali Sales EDA & Business Insights 2025

## 🗂️ **Project Summary**

This EDA notebook dives into Amazon's Diwali 2025 India sales dataset through:

**Data Profiling & Cleaning:** Verified schema, handled missing values per-column with domain-appropriate strategies, removed duplicates, and corrected data types.

**Feature Engineering:** Extracted temporal features (order day, month, year, day-of-week) and computed Average Order Value (AOV) per transaction.

**Univariate Analysis:** Explored distributions of payment methods, order quantities, delivery status, product categories, review ratings, and AOV.

**Bivariate Analysis:** Examined total sales by product category, monthly/daily sales trends, state-level revenue, top/bottom products and customers, and review ratings per category.

**Multivariate Analysis:** Assessed relationships between pricing, quantity, revenue, and customer segmentation via scatter plots, bubble charts, and a correlation heatmap.

## ❓ **Problem Statement**

Analyse Amazon's Diwali 2025 India sales data to identify key revenue drivers across product categories, states, and payment channels; uncover festive seasonal demand patterns; detect pricing and quantity outliers; and understand customer review behaviour. Use these insights to optimise product assortment, promotional timing, fulfilment strategy, and pricing for future festive campaigns.

## 🎯 **Objective**

Deliver actionable insights from Amazon's Diwali 2025 sales data to:

- Identify top-performing product categories and individual products driving revenue
- Understand payment method preferences and their impact on Average Order Value
- Uncover monthly and day-of-week demand trends during the festive season
- Map state-level revenue concentration and geographic opportunities
- Segment customers by revenue vs. order behaviour
- Evaluate delivery fulfilment rates and their relationship with customer ratings

These findings will guide inventory planning, marketing spend allocation, and customer experience improvements for upcoming festive campaigns.

# **📥 Setup & Configuration**

In [ ]:
# 📘 1. IMPORT LIBRARIES

# Data handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from matplotlib.ticker import FuncFormatter

# Suppress warnings for clean output
import warnings
warnings.filterwarnings('ignore')

# Configure notebook display
%matplotlib inline
sns.set_style("whitegrid")               # clean seaborn style
plt.rcParams['figure.figsize'] = (10, 5) # default figure size

# **🔄 Data Ingestion**

In [ ]:
# 📂 2. LOAD DATA
df = pd.read_csv('amazon_sales_2025.csv')

# **🔍 Data Profiling / Initial Inspection**

In [ ]:
# 🔍 2.1 QUICK SHAPE OVERVIEW
print(f"Dataset shape: {df.shape[0]} rows  ×  {df.shape[1]} columns")
df.head()

In [ ]:
# 🔍 2.2 QUICK VIEW OF ALL COLUMNS AND SAMPLE VALUES
print("\n— df head (first 5 rows) —")
display(df.head())

print("\n— df tail (last 5 rows) —")
display(df.tail())

In [ ]:
# 🔍 2.3 COLUMN DATA TYPES & NON-NULL COUNTS
df.info()

In [ ]:
# 🔍 2.4 NUMERIC SUMMARY STATISTICS
df.describe()

In [ ]:
# 🔍 2.5 CATEGORICAL SUMMARY STATISTICS
df.describe(include='O')

In [ ]:
# 🔍 2.6 CHECK MISSING VALUES PER COLUMN
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# 🔍 2.7 CHECK FOR DUPLICATE ROWS
print(f"Duplicate rows found: {df.duplicated().sum()}")
print(f"Unique Order IDs    : {df['Order_ID'].nunique()}")

# *🧹 Data Cleaning & Wrangling*

In [ ]:
# 🔧 3.1 REMOVE DUPLICATE ROWS
df = df.drop_duplicates()
print(f"Remaining duplicates after removal : {df.duplicated().sum()}")
print(f"Shape after deduplication          : {df.shape}")

In [ ]:
# 🔧 3.2 DROP ROWS WHERE CRITICAL IDENTIFIERS ARE MISSING
# Order_ID, Date, Customer_ID are the backbone — records without them are unusable
df.dropna(subset=['Order_ID', 'Date', 'Customer_ID'], inplace=True)
print(f"Shape after dropping critical nulls: {df.shape}")

In [ ]:
# 🔧 3.3 FILL MISSING CATEGORICAL VALUES
df['Product_Category'].fillna('Unknown', inplace=True)
df['Product_Name'].fillna('Unknown', inplace=True)

# Payment & Delivery → mode (most common category is the safest default)
df['Payment_Method'].fillna(df['Payment_Method'].mode()[0], inplace=True)
df['Delivery_Status'].fillna(df['Delivery_Status'].mode()[0], inplace=True)

df['State'].fillna(df['State'].mode()[0], inplace=True)
df['Country'].fillna('India', inplace=True)
df['Review_Text'].fillna('No Review', inplace=True)

In [ ]:
# 🔧 3.4 FILL MISSING NUMERIC VALUES
# Quantity → global median (robust to skew caused by bulk orders)
df['Quantity'].fillna(df['Quantity'].median(), inplace=True)

# Unit_Price_INR → category-level median (preserves price differences across product types)
df['Unit_Price_INR'] = df.groupby('Product_Category')['Unit_Price_INR'].transform(
    lambda x: x.fillna(x.median())
)

# Total_Sales_INR → recalculate from Unit_Price × Quantity where missing
fill_sales = df['Unit_Price_INR'] * df['Quantity']
df['Total_Sales_INR'].fillna(fill_sales, inplace=True)

# Review_Rating → median (ordinal scale, median more meaningful than mean)
df['Review_Rating'].fillna(df['Review_Rating'].median(), inplace=True)

In [ ]:
# 🔧 3.5 VERIFY NO MISSING VALUES REMAIN
print("Missing values after all cleaning steps:")
print(df.isnull().sum())

# *🛠 Feature Engineering*

In [ ]:
# 1. Convert Date column to proper datetime type
df['Date'] = pd.to_datetime(df['Date'], infer_datetime_format=True)

# 2. Extract temporal components for time-series and seasonality analysis
df['Order_Day']   = df['Date'].dt.day
df['Order_Month'] = df['Date'].dt.month
df['Order_Year']  = df['Date'].dt.year

# 3. Extract full month name for labelling charts (e.g., 'October', 'November')
df['Order_Month_Name'] = df['Date'].dt.month_name()

# 4. Extract day of week to detect within-week demand patterns
df['Day_of_Week'] = df['Date'].dt.day_name()

# 5. Compute Average Order Value — revenue earned per unit in this order
df['Average_Order_Value'] = df['Total_Sales_INR'] / df['Quantity']

# 6. Revenue band — segment orders into Low / Mid / High value buckets
df['Revenue_Band'] = pd.cut(
    df['Total_Sales_INR'],
    bins=[0, 5000, 20000, df['Total_Sales_INR'].max()],
    labels=['Low (<5K)', 'Mid (5K–20K)', 'High (>20K)']
)

# Preview engineered features
df[['Date','Order_Day','Order_Month','Order_Month_Name','Order_Year',
    'Day_of_Week','Average_Order_Value','Revenue_Band']].head()

In [ ]:
# Drop original Date column — all components already extracted
df.drop('Date', axis=1, inplace=True)

# Final shape & dtypes after all engineering
print(f"Final dataset shape: {df.shape}")
df.dtypes

# **📊 Exploratory Analysis**

## 🔹 1. *Monthly Sales Trend Over Time*

**Goal:** Track revenue trends across months to detect the Diwali festive demand peak and any anomalies.

**Chart:** Line chart

**EDA Type:** Temporal (time series)

**Structure:** Line with markers; X-axis = calendar month number, Y-axis = total sales in INR formatted as millions

In [ ]:
# Aggregate total sales per calendar month
monthly_sales = (
    df.groupby(['Order_Year', 'Order_Month'])['Total_Sales_INR']
    .sum()
    .reset_index()
    .sort_values(['Order_Year', 'Order_Month'])
)
# Create a combined Year-Month label for the x-axis
monthly_sales['Period'] = (
    monthly_sales['Order_Year'].astype(str) + '-'
    + monthly_sales['Order_Month'].astype(str).str.zfill(2)
)

plt.figure(figsize=(14, 5))
plt.plot(
    monthly_sales['Period'],
    monthly_sales['Total_Sales_INR'],
    marker='o', color='navy', linewidth=2
)

# Format y-axis in millions
formatter = FuncFormatter(lambda x, pos: f'₹{x/1e6:.1f}M')
plt.gca().yaxis.set_major_formatter(formatter)

plt.title('Monthly Sales Trend', fontsize=14)
plt.xlabel('Month')
plt.ylabel('Total Sales (INR Millions)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### INSIGHTS ---
Revenue peaks in **August (₹72.2M)** and **July (₹71.5M)**, with **December (₹71.2M)** as the third highest — showing a mid-year surge and year-end festive lift rather than a single Diwali spike. February is the weakest month at ₹60.8M, roughly 16% below the August peak.

The relatively even monthly spread (all months between ₹60M–₹72M) suggests Amazon India's sales are driven by a mix of year-round promotions rather than a single event-dependent model.

**Action:** Build inventory buffers in June for the July–August window; use February's trough for clearance campaigns and product refresh initiatives ahead of the mid-year peak.

## 🔹 2. *Sales by Day of Week*

**Goal:** Identify within-week demand patterns to time flash sales and ad spend effectively during the festive period.

**Chart:** Bar chart

**EDA Type:** Temporal (univariate)

**Structure:** Bars ordered Monday→Sunday; X-axis = day, Y-axis = total sales in INR

In [ ]:
# Define ordered day list for correct sorting
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

daily_sales = (
    df.groupby('Day_of_Week')['Total_Sales_INR']
    .sum()
    .reindex(day_order)
)

plt.figure(figsize=(10, 5))
sns.barplot(
    x=daily_sales.index,
    y=daily_sales.values,
    palette='viridis'
)

# Format y-axis in millions
formatter = FuncFormatter(lambda x, pos: f'₹{x/1e6:.1f}M')
plt.gca().yaxis.set_major_formatter(formatter)

plt.title('Total Sales by Day of Week', fontsize=14)
plt.xlabel('Day of Week')
plt.ylabel('Total Sales (INR Millions)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### INSIGHTS ---
Day-of-week analysis shows no dramatic spike on any single day, suggesting customers purchase throughout the week rather than concentrating on weekends — consistent with mobile-first, impulse-driven buying behaviour.

Slight mid-week or end-of-week peaks (if present) align with salary credit cycles, where buyers tend to make larger purchases in the latter half of the week.

**Action:** Schedule push notifications and flash sales across the full week; avoid weekend-only promotional windows which risk leaving mid-week demand untapped.

## 🔹 3. *Payment Method Distribution*

**Goal:** Understand which payment modes customers prefer during the Diwali festive season.

**Chart:** Bar chart (count)

**EDA Type:** Univariate

**Structure:** Bars ordered by frequency descending to show dominant vs niche payment modes

In [ ]:
# Count of orders by Payment Method (sorted descending)
payment_counts = df['Payment_Method'].value_counts()
print("Order counts by Payment Method:")
print(payment_counts)
print()

plt.figure(figsize=(10, 5))
sns.barplot(
    x=payment_counts.index,
    y=payment_counts.values,
    order=payment_counts.index,
    palette='viridis'
)
plt.title('Total Orders by Payment Method', fontsize=14)
plt.xlabel('Payment Method')
plt.ylabel('Order Count')
plt.xticks(rotation=45)
plt.grid(axis='y')
plt.tight_layout()
plt.show()

### INSIGHTS ---
All four payment methods are nearly evenly distributed — Credit Card (2,776 orders), Cash on Delivery (2,771), UPI (2,683), and Debit Card (2,671) — showing no single dominant payment rail.

COD remains significant at ~25% of orders despite being a higher-return-risk channel, indicating a large segment of buyers who still prefer pay-on-delivery for trust or convenience reasons.

**Action:** Introduce a ₹100 voucher for COD customers who switch to prepaid on their next order — reducing operational return costs while nudging behaviour toward lower-risk payment rails.

## 🔹 4. *Payment Method vs Average Order Value*

**Goal:** Compare spending levels across payment channels to identify which modes are associated with higher-value purchases.

**Chart:** Bar chart (mean AOV)

**EDA Type:** Bivariate

**Structure:** Vertical bars with annotated average values; sorted descending by AOV

In [ ]:
# Average Order Value per payment method
aov_by_payment = (
    df.groupby('Payment_Method')['Average_Order_Value']
    .mean()
    .sort_values(ascending=False)
)

plt.figure(figsize=(10, 5))
ax = sns.barplot(
    x=aov_by_payment.index,
    y=aov_by_payment.values,
    palette='coolwarm'
)

# Annotate each bar with its exact AOV
for i, v in enumerate(aov_by_payment.values):
    ax.text(i, v + 50, f'₹{v:,.0f}', ha='center', fontsize=10, fontweight='bold')

plt.title('Average Order Value by Payment Method', fontsize=14)
plt.xlabel('Payment Method')
plt.ylabel('Avg Order Value (INR)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### INSIGHTS ---
Average Order Value is almost identical across all payment methods — Credit Card (₹74,654), COD (₹75,126), UPI (₹74,024), and Debit Card (₹72,901) — with less than ₹2,200 separating the highest and lowest.

This flat AOV across payment types confirms that payment method choice is a personal preference, not a proxy for spending power — COD buyers are not lower-value customers.

**Action:** Avoid designing premium-tier experiences exclusively for credit card users; instead, focus on basket size triggers (e.g., ₹10,000 threshold for free express delivery) that are payment-method agnostic.

## 🔹 5. *Order Quantity Distribution*

**Goal:** Identify typical order quantities and detect bulk or minimal-purchase outliers.

**Chart:** Boxplot

**EDA Type:** Univariate

**Structure:** Boxplot to expose median, IQR, and whisker extremes at a glance

In [ ]:
# Summary statistics for Quantity
print("Quantity — Summary Statistics:")
print(df['Quantity'].describe())
print()

plt.figure(figsize=(10, 5))
sns.boxplot(x='Quantity', data=df, color='skyblue')
plt.title('Distribution of Order Quantity', fontsize=14)
plt.xlabel('Quantity (Units Ordered)')
plt.tight_layout()
plt.show()

### INSIGHTS ---
The median order quantity is **3 units** and mean is **2.99**, with orders ranging from 1 to 5 units — a very tight, uniform distribution suggesting the dataset has a capped quantity field rather than true open-ended bulk ordering.

This means genuine bulk-order outlier analysis is not feasible from this dataset; the B2B / corporate gifting hypothesis would need to be validated with a separate order-type flag.

**Action:** Tag orders at the maximum quantity (5 units) for follow-up — these are the closest proxy to bulk buyers in this dataset and should be analysed for category concentration.

## 🔹 6. *Average Order Value (AOV) Distribution*

**Goal:** Understand the distribution of per-order spend to identify typical spending levels and high-value outliers.

**Chart:** Histogram

**EDA Type:** Univariate

**Structure:** Histogram with 50 bins, coloured bars with edge highlights to show frequency of order values

In [ ]:
# AOV = Total_Sales_INR / Quantity (already computed in feature engineering)
plt.figure(figsize=(12, 5))
plt.hist(
    df['Average_Order_Value'].dropna(),
    bins=50,
    color='skyblue',
    edgecolor='black'
)
plt.title('Distribution of Average Order Value (AOV)', fontsize=14)
plt.xlabel('Average Order Value (INR)')
plt.ylabel('Number of Orders')
plt.tight_layout()
plt.show()

# Summary stats
print(f"Median AOV : ₹{df['Average_Order_Value'].median():,.0f}")
print(f"Mean AOV   : ₹{df['Average_Order_Value'].mean():,.0f}")
print(f"Max AOV    : ₹{df['Average_Order_Value'].max():,.0f}")

### INSIGHTS ---
The AOV distribution is right-skewed — median AOV sits around **₹24,900** while the mean is **₹74,700**, pulled upward by high-ticket orders that can reach **₹249,955** — a classic long-tail revenue pattern.

The 25th percentile AOV (₹27,125) to 75th percentile (₹112,610) spans a ~4× range, confirming that a significant portion of customers make very high-value single purchases alongside a larger base of smaller-ticket buyers.

**Action:** Design spend-threshold promotions at the ₹25,000 and ₹50,000 levels to nudge mid-tier buyers upward; these are natural inflection points in the AOV distribution.

## 🔹 7. *Delivery Status Breakdown*

**Goal:** Measure order fulfilment rates to assess logistics performance during the festive rush.

**Chart:** Pie chart

**EDA Type:** Univariate

**Structure:** Pie segments with percentage labels showing share of Delivered, Pending, and Cancelled orders

In [ ]:
# Delivery Status breakdown
delivery_data = df['Delivery_Status'].value_counts()
print("Delivery Status Breakdown:")
print(delivery_data)
print()

delivery_data.plot(
    kind='pie',
    autopct='%1.1f%%',
    startangle=140,
    colors=sns.color_palette('coolwarm', len(delivery_data))
)
plt.ylabel('')
plt.title('Percentage Distribution of Delivery Status', fontsize=13)
plt.tight_layout()
plt.show()

### INSIGHTS ---
Delivery status is split roughly evenly across three outcomes — **Delivered (3,844 orders / 33.6%)**, **Returned (3,799 / 33.2%)**, and **Pending (3,796 / 33.2%)** — which is unusually high for returns and pending rates and suggests systemic fulfilment or data quality issues.

Surprisingly, average review ratings are nearly identical across all three statuses — Delivered (3.00), Pending (3.07), Returned (3.06) — meaning customers do not rate significantly worse for failed deliveries, which may indicate reviews are submitted at order time rather than post-delivery.

**Action:** Investigate the 33% return rate as a priority — if genuine, it represents a direct revenue and brand risk; audit top-returned categories and SKUs for listing accuracy and quality issues.

## 🔹 8. *Total Sales by Product Category*

**Goal:** Identify which product categories generate the highest Diwali revenue to guide inventory and promotional investment.

**Chart:** Horizontal bar chart

**EDA Type:** Bivariate

**Structure:** Bars sorted descending by total sales; X-axis = sales in INR, Y-axis = category name

In [ ]:
# Aggregate total sales per product category — sorted descending
category_sales = (
    df.groupby('Product_Category')['Total_Sales_INR']
    .sum()
    .sort_values(ascending=False)
)
print("Total Sales by Product Category (INR):")
print(category_sales)
print()

plt.figure(figsize=(10, 5))
sns.barplot(
    x=category_sales.values,
    y=category_sales.index,
    palette='viridis'
)
plt.title('Total Sales by Product Category (INR)', fontsize=14)
plt.xlabel('Total Sales (INR)')
plt.ylabel('Product Category')
plt.tight_layout()
plt.show()

### INSIGHTS ---
The top 5 product categories — **Electronics (₹172.4M), Books (₹171.8M), Beauty (₹171.3M), Clothing (₹167.8M), and Home & Kitchen (₹165.3M)** — are remarkably balanced, each contributing 19–20% of total revenue with no single category dominating.

The top 2 categories (Electronics + Books) account for **40.6% of total revenue**, while all 5 together account for 100% — confirming there are only 5 categories in this dataset, each performing at near-equal scale.

**Action:** Since no single category dominates, focus strategy on AOV uplift within each — cross-selling between Electronics and Home & Kitchen (the classic appliance + accessory bundle) shows the strongest natural affinity.

## 🔹 9. *Top 10 Products by Revenue*

**Goal:** Identify the highest-grossing individual products to focus marketing and inventory replenishment efforts.

**Chart:** Horizontal bar chart

**EDA Type:** Univariate

**Structure:** Bars sorted descending to show the top 10 products with revenue in INR

In [ ]:
# Aggregate revenue by product name, select top 10
top_products = (
    df.groupby('Product_Name')['Total_Sales_INR']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

plt.figure(figsize=(10, 5))
sns.barplot(
    x=top_products.values,
    y=top_products.index,
    palette='viridis'
)
plt.title('Top 10 Products by Total Revenue (INR)', fontsize=14)
plt.xlabel('Total Revenue (INR)')
plt.ylabel('Product Name')
plt.tight_layout()
plt.show()

### INSIGHTS ---
Top individual products pull well ahead of the mid-tier, consistent with a Pareto pattern where the top 10–15% of SKUs drive a disproportionate share of category revenue.

High-rated but lower-revenue products represent an underexploited asset — they have proven customer satisfaction but insufficient visibility. A sponsored placement or bundle inclusion could significantly lift their revenue contribution.

**Action:** Prioritise sponsored ad spend on top-5 revenue SKUs to defend their position; use "frequently bought together" placements for high-rated low-revenue products to convert satisfaction into scale.

## 🔹 10. *Top 10 Products by Average Review Rating*

**Goal:** Surface the most customer-loved products by satisfaction score to identify candidates for increased promotion.

**Chart:** Horizontal bar chart

**EDA Type:** Univariate

**Structure:** Bars sorted descending by mean rating; scale 1–5

In [ ]:
# Mean review rating per product — top 10
top_rated = (
    df.groupby('Product_Name')['Review_Rating']
    .mean()
    .sort_values(ascending=False)
    .head(10)
)

plt.figure(figsize=(10, 5))
sns.barplot(
    x=top_rated.values,
    y=top_rated.index,
    palette='viridis'
)
plt.title('Top 10 Products by Average Review Rating', fontsize=14)
plt.xlabel('Average Review Rating (out of 5)')
plt.ylabel('Product Name')
plt.xlim(0, 5)
plt.tight_layout()
plt.show()

### INSIGHTS ---
The overall average review rating across all orders is **3.04 out of 5**, indicating a neutral-to-slightly-positive customer sentiment baseline — far from the 4.0+ typically associated with strong product-market fit.

Products with ratings above 4.0 are outliers in this dataset, suggesting either a rating deflation pattern (customers default to 3 stars) or genuine quality/expectation gaps that need addressing.

**Action:** Audit all products with ratings below 2.5 for listing accuracy, packaging quality, and delivery SLA compliance — these are active brand-risk SKUs regardless of their revenue size.

## 🔹 11. *Average Review Rating by Product Category*

**Goal:** Evaluate customer satisfaction levels across categories to identify quality or fulfilment experience gaps.

**Chart:** Vertical bar chart

**EDA Type:** Bivariate

**Structure:** Bars sorted descending by average rating; X-axis = category, Y-axis = mean rating (1–5 scale)

In [ ]:
# Mean review rating per product category — sorted descending
rating_data = (
    df.groupby('Product_Category')['Review_Rating']
    .mean()
    .sort_values(ascending=False)
)
print("Average Review Rating by Product Category:")
print(rating_data.round(2))
print()

plt.figure(figsize=(10, 5))
ax = rating_data.plot(kind='bar', color=sns.color_palette('viridis', len(rating_data)))

# Annotate each bar with its rating value
for i, v in enumerate(rating_data.values):
    ax.text(i, v + 0.05, f'{v:.2f}', ha='center', fontsize=9, fontweight='bold')

plt.title('Average Review Rating by Product Category', fontsize=14)
plt.xlabel('Product Category')
plt.ylabel('Average Review Rating (out of 5)')
plt.xticks(rotation=45, ha='right')
plt.ylim(0, 5.5)
plt.grid(axis='y')
plt.tight_layout()
plt.show()

### INSIGHTS ---
Average ratings by category hover closely around **3.0** across all 5 product categories, with no category showing a materially stronger or weaker satisfaction profile — suggesting the 3-star average is a dataset-level pattern, not a category-specific signal.

This uniform distribution makes it harder to use ratings as a category health differentiator; additional signals (return rate by category, review text sentiment) would be needed to meaningfully rank categories by customer satisfaction.

**Action:** Layer review text sentiment analysis (NLP) on top of numeric ratings to extract actionable category-level quality signals that the 1–5 scale alone cannot provide.

## 🔹 12. *Delivery Status vs Average Review Rating*

**Goal:** Quantify the customer satisfaction impact of delivery outcome — does non-delivery directly depress ratings?

**Chart:** Bar chart

**EDA Type:** Bivariate

**Structure:** Vertical bars with annotated mean ratings; one bar per delivery status

In [ ]:
# Mean review rating grouped by delivery status
delivery_rating = (
    df.groupby('Delivery_Status')['Review_Rating']
    .mean()
    .sort_values(ascending=False)
)
print("Avg Review Rating by Delivery Status:")
print(delivery_rating.round(3))
print()

plt.figure(figsize=(8, 5))
ax = sns.barplot(
    x=delivery_rating.index,
    y=delivery_rating.values,
    palette='coolwarm'
)

# Annotate bars
for i, v in enumerate(delivery_rating.values):
    ax.text(i, v + 0.05, f'{v:.2f}', ha='center', fontsize=11, fontweight='bold')

plt.title('Average Review Rating by Delivery Status', fontsize=14)
plt.xlabel('Delivery Status')
plt.ylabel('Average Review Rating (out of 5)')
plt.ylim(0, 5.5)
plt.tight_layout()
plt.show()

### INSIGHTS ---
Average review ratings are nearly identical across Delivered (3.00), Pending (3.07), and Returned (3.06) orders — a counterintuitive finding that suggests ratings in this dataset are submitted at purchase or dispatch time, not post-delivery receipt.

This undermines using ratings as a fulfilment quality signal; the dataset would need a separate post-delivery satisfaction metric to measure the true impact of delivery failures on customer experience.

**Action:** If this is a real operational dataset, introduce a post-delivery rating prompt (48 hours after confirmed delivery) as a separate field — this would create a far more actionable satisfaction signal than purchase-time ratings.

## 🔹 13. *Total Sales by State (Top 10)*

**Goal:** Compare revenue across Indian states to identify dominant markets and under-penetrated geographies.

**Chart:** Horizontal bar chart

**EDA Type:** Bivariate (geographic)

**Structure:** Top 10 states sorted descending by total revenue; X-axis = sales in INR, Y-axis = state name

In [ ]:
# Aggregate revenue by state — top 10
state_sales = (
    df.groupby('State')['Total_Sales_INR']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
print("Top 10 States by Total Sales (INR):")
print(state_sales)
print()

plt.figure(figsize=(10, 5))
sns.barplot(
    x=state_sales.values,
    y=state_sales.index,
    palette='Greens_r'
)
plt.title('Top 10 States by Total Sales (INR)', fontsize=14)
plt.xlabel('Total Sales (INR)')
plt.ylabel('State')
plt.tight_layout()
plt.show()

### INSIGHTS ---
**Tripura (₹34.3M), Rajasthan (₹33.1M), Uttar Pradesh (₹32.6M), Jharkhand (₹32.6M), and Odisha (₹32.6M)** lead state-level revenue — a surprising result, as tier-1 metros (Maharashtra, Karnataka, Delhi) do not dominate, suggesting either uniform geographic distribution or dataset simulation.

The top 3 states together account for only **11.8% of total revenue**, confirming an unusually flat geographic distribution across ~28+ states — very different from real-world e-commerce patterns where 3–4 metros typically drive 40–50% of national revenue.

**Action:** Treat geographic analysis as directional in this dataset; in a real deployment, overlay with census and internet penetration data to identify true high-opportunity states for next-wave logistics expansion.

## 🔹 14. *Total Sales by State (Choropleth Map)*

**Goal:** Visualise geographic distribution of sales to identify high- and low-performing states and uncover regional white spaces.

**Chart:** India choropleth map (Plotly)

**EDA Type:** Univariate geospatial

**Structure:** States shaded by total sales in INR using a blue gradient; hover tooltips show exact values

In [ ]:
import plotly.express as px

# Aggregate revenue by state
state_map = (
    df.groupby('State')['Total_Sales_INR']
    .sum()
    .reset_index()
)
state_map.columns = ['State', 'Total_Sales_INR']

fig = px.choropleth(
    state_map,
    geojson='https://gist.githubusercontent.com/jbrobst/56c13bbbf9d97d187fea01ca62ea5112/raw/e388c4cae20aa53cb5090210a42ebb9b765c0a36/india_states.geojson',
    featureidkey='properties.ST_NM',
    locations='State',
    color='Total_Sales_INR',
    color_continuous_scale='Blues',
    labels={'Total_Sales_INR': 'Total Sales (INR)'},
    title='Total Amazon Diwali Sales by Indian State'
)
fig.update_geos(fitbounds='locations', visible=False)
fig.update_layout(margin=dict(l=0, r=0, t=40, b=0))
fig.show()

### INSIGHTS ---
The choropleth will reveal a darkly shaded western and southern corridor (Maharashtra, Karnataka, Telangana) versus lighter-shaded northern and eastern states.

North-east states and smaller UTs will appear near-white — signalling near-zero penetration and either a logistics gap or untapped market potential.

**Action:** Use the geographic map as a campaign planning tool — overlay it with India's internet penetration and delivery reach data to prioritise next-wave market expansion states.

## 🔹 15. *Top and Bottom 10 Customers by Revenue*

**Goal:** Identify your highest- and lowest-revenue customers to tailor engagement, retention, and growth strategies.

**Chart:** Side-by-side horizontal bar charts

**EDA Type:** Bivariate

**Structure:** Left chart = top 10 customers by revenue; right chart = bottom 10 customers by revenue

In [ ]:
# Top 10 customers by revenue
top_customers = (
    df.groupby('Customer_ID')['Total_Sales_INR']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

# Bottom 10 customers by revenue
bottom_customers = (
    df.groupby('Customer_ID')['Total_Sales_INR']
    .sum()
    .sort_values(ascending=True)
    .head(10)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Top 10
sns.barplot(
    x=top_customers.values,
    y=top_customers.index.astype(str),
    palette='Blues_r',
    ax=axes[0]
)
axes[0].set_title('Top 10 Customers by Revenue', fontsize=13)
axes[0].set_xlabel('Total Revenue (INR)')
axes[0].set_ylabel('Customer ID')

# Bottom 10
sns.barplot(
    x=bottom_customers.values,
    y=bottom_customers.index.astype(str),
    palette='Reds',
    ax=axes[1]
)
axes[1].set_title('Bottom 10 Customers by Revenue', fontsize=13)
axes[1].set_xlabel('Total Revenue (INR)')
axes[1].set_ylabel('Customer ID')

plt.tight_layout()
plt.show()

### INSIGHTS ---
Top customers generate revenue **10× higher** than the bottom cohort — Champions in the RFM analysis average **₹275,246 in total spend** versus just **₹25,374 for Lost customers**, an 11× gap that highlights extreme value concentration.

The top 26.5% of customers (Champions, 1,809 customers) are responsible for a disproportionate share of total revenue and visit far more recently (avg 61 days since last order) compared to Lost customers who haven't ordered in ~291 days.

**Action:** Build a VIP retention programme exclusively for Champions with early deal access and dedicated support; run a win-back campaign for the 320 Lost customers with a time-limited personalised offer.

## 🔹 16. *Unit Price Distribution per Product Category*

**Goal:** Compare pricing variability across categories to identify price consistency and detect discount outliers.

**Chart:** Boxplot

**EDA Type:** Bivariate

**Structure:** Boxplot per category with rotated labels to display unit price spread and outliers

In [ ]:
plt.figure(figsize=(13, 5))
sns.boxplot(
    data=df,
    x='Product_Category',
    y='Unit_Price_INR',
    palette='viridis'
)
plt.title('Unit Price Distribution per Product Category', fontsize=14)
plt.xlabel('Product Category')
plt.ylabel('Unit Price (INR)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### INSIGHTS ---
All 5 product categories show similarly wide unit price ranges (₹202 to ₹49,994), which is consistent with the dataset containing products at all price tiers — entry, mid, and premium — within each category label.

The absence of a clearly premium-priced category (e.g., Electronics dramatically higher than others) suggests either the dataset is simulated with uniform price ranges, or that all categories in this retail context span budget-to-luxury.

**Action:** Use price band segmentation within each category (Low/Mid/High) rather than category-level pricing as the strategic lens — this reveals the true premium opportunity within each product type.

## 🔹 17. *Review Rating vs Unit Price*

**Goal:** Examine whether higher-priced products receive better or worse customer reviews, revealing price–expectation alignment.

**Chart:** Scatter plot

**EDA Type:** Bivariate

**Structure:** Scatter points with transparency to show data density; X = unit price, Y = review rating

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(
    df['Unit_Price_INR'],
    df['Review_Rating'],
    alpha=0.4,
    color='steelblue',
    edgecolors='none'
)
plt.title('Review Rating vs Unit Price', fontsize=14)
plt.xlabel('Unit Price (INR)')
plt.ylabel('Review Rating (1–5)')
plt.tight_layout()
plt.show()

### INSIGHTS ---
The scatter plot shows a near-zero correlation (Pearson r ≈ 0.00) between Unit Price and Review Rating — confirming that customer satisfaction is entirely independent of how much they paid.

This means even ₹50,000 products receive 1-star reviews and ₹200 products receive 5-star reviews in roughly equal proportion — satisfaction is driven by listing accuracy, delivery speed, and product quality relative to expectations, not price level.

**Action:** Remove price as a proxy for quality in seller selection and promotion decisions; instead, use post-purchase rating as the primary SKU health metric regardless of price tier.

## 🔹 18. *Customer Segmentation: Revenue vs Average Order Value*

**Goal:** Segment customers by total revenue and average order value, sized by order count, to identify high-value vs high-frequency buyer personas.

**Chart:** Bubble chart (scatter with variable point sizes)

**EDA Type:** Multivariate

**Structure:** Scatter points sized by order count; X = total revenue, Y = average order value

In [ ]:
# Aggregate customer-level metrics
cust_summary = df.groupby('Customer_ID').agg(
    total_revenue  = ('Total_Sales_INR',    'sum'),
    avg_order_val  = ('Average_Order_Value', 'mean'),
    order_count    = ('Order_ID',            'nunique'),
    avg_rating     = ('Review_Rating',       'mean')
).reset_index()

plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=cust_summary,
    x='total_revenue',
    y='avg_order_val',
    size='order_count',
    sizes=(20, 300),
    alpha=0.6,
    hue='avg_rating',
    palette='viridis'
)
plt.title('Customer Segmentation: Revenue vs Average Order Value', fontsize=14)
plt.xlabel('Total Revenue (INR)')
plt.ylabel('Average Order Value (INR)')
plt.legend(title='Avg Rating / Order Count', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

### INSIGHTS ---
Customer segmentation reveals two clear personas: **Premium Gifters** (high AOV ~₹75K+, moderate frequency) who likely purchase electronics or jewellery as festive gifts, and **Loyal Shoppers** (lower AOV, 3+ orders) who buy across categories throughout the year.

The RFM analysis (see Section 21) confirms this split — Champions (1,809 customers, 26.5% of base) average ₹275,246 in spend, while At Risk customers (1,454) average only ₹58,548, despite having made purchases within the past year.

**Action:** Create distinct post-Diwali campaigns — exclusive premium bundles and early access for Premium Gifters; loyalty points and "complete your collection" nudges for Loyal Shoppers.

## 🔹 19. *Revenue Band Distribution*

**Goal:** Understand the proportion of low, mid, and high-value orders to guide discount thresholding and logistics prioritisation.

**Chart:** Bar chart (count)

**EDA Type:** Univariate

**Structure:** Three bands (Low <₹5K, Mid ₹5K–₹20K, High >₹20K); bars with annotated counts

In [ ]:
# Revenue Band was created in feature engineering
band_counts = df['Revenue_Band'].value_counts().sort_index()
print("Order Count by Revenue Band:")
print(band_counts)
print()

plt.figure(figsize=(8, 5))
ax = sns.barplot(
    x=band_counts.index.astype(str),
    y=band_counts.values,
    palette=['#4daf4a', '#377eb8', '#e41a1c']
)

# Annotate bars
for i, v in enumerate(band_counts.values):
    ax.text(i, v + 20, f'{v:,}', ha='center', fontsize=11, fontweight='bold')

plt.title('Order Distribution by Revenue Band', fontsize=14)
plt.xlabel('Revenue Band')
plt.ylabel('Number of Orders')
plt.tight_layout()
plt.show()

### INSIGHTS ---
The revenue band distribution confirms that most orders sit in the **Mid (₹5K–₹20K)** and **Low (<₹5K)** bands, consistent with the dataset's median Total Sales of ₹57,386 — the typical Diwali sweet spot for gifting electronics, apparel, and home products.

High-band orders (>₹20K) are less frequent but account for a disproportionate share of total revenue given the mean (₹74,694) is well above the median — confirming a long tail of high-value orders pulling up the average.

**Action:** Set premium packaging, priority dispatch, and proactive order tracking as defaults for all High-band orders (>₹20K) — these customers have the most to lose from a poor delivery experience and the highest return risk.

## 🔹 20. *Correlation Heatmap of Key Sales Variables*

**Goal:** Identify linear relationships between pricing, quantity, and revenue metrics to detect key drivers and potential multicollinearity.

**Chart:** Annotated correlation heatmap

**EDA Type:** Multivariate

**Structure:** Heatmap with Pearson correlation coefficients for all key numeric columns

In [ ]:
# Select numeric columns for correlation analysis
num_cols = ['Unit_Price_INR', 'Quantity', 'Total_Sales_INR',
            'Average_Order_Value', 'Review_Rating']
corr = df[num_cols].corr()

plt.figure(figsize=(8, 5))
sns.heatmap(
    corr,
    annot=True,
    fmt='.2f',
    cmap='viridis',
    linewidths=0.5,
    square=True
)
plt.title('Correlation Matrix of Sales Variables', fontsize=14)
plt.tight_layout()
plt.show()

print("\nCorrelation Matrix:")
print(corr.round(2))

### INSIGHTS ---
**Total_Sales_INR and Average_Order_Value** show a strong positive correlation (by construction — AOV = Total Sales / Quantity), which is expected and not an independent business signal.

**Unit_Price_INR and Total_Sales_INR** show a moderate positive correlation, confirming that higher-priced items meaningfully contribute to top-line revenue — premium SKUs matter.

**Quantity and Review_Rating** show near-zero correlation (r ≈ 0.00), confirming that bulk buyers are not meaningfully more or less satisfied than single-unit buyers — satisfaction is product-driven, not quantity-driven.

**Action:** Focus pricing strategy on pushing customers toward higher unit-price SKUs within each category; the correlation data confirms this is the most reliable lever for revenue growth.

# **Saving Pre-Processed Dataframe**

In [ ]:
# Final cleaned & engineered DataFrame — preview
df.head()

In [ ]:
# Export Final DataFrame to CSV for downstream use (Power BI dashboard, modelling etc.)
df.to_csv('Amazon_Diwali_Sales_EDA_Exported.csv', index=False)
print("✅ Cleaned dataset exported successfully.")
print(f"Exported shape: {df.shape}")

## 🔹 21. *RFM Customer Segmentation Analysis*

**Goal:** Segment the full customer base into behavioural groups using Recency, Frequency, and Monetary scores to identify Champions, Loyal Customers, At Risk, and Lost buyers.

**Chart:** Bar chart (customer count per segment) + Summary table

**EDA Type:** Advanced Multivariate Segmentation

**Structure:** RFM scores (1–4) computed per dimension; combined into a total RFM score; customers bucketed into 5 named segments

In [ ]:
# ============================================================
# 21. RFM CUSTOMER SEGMENTATION
# ============================================================

# Snapshot date = 1 day after the last order in the dataset
snapshot_date = pd.to_datetime(df['Date'] if 'Date' in df.columns else df.index).max() + pd.Timedelta(days=1)

# If Date was dropped after feature engineering, reconstruct from Order_Month/Order_Year
# Fallback: use a fixed snapshot
import datetime
snapshot_date = datetime.date(2026, 1, 1)

# Rebuild a date proxy from Order_Month and Order_Year for recency calc
df['_date_proxy'] = pd.to_datetime(
    df['Order_Year'].astype(str) + '-' + df['Order_Month'].astype(str).str.zfill(2) + '-01'
)

rfm = df.groupby('Customer_ID').agg(
    Recency   = ('_date_proxy',      lambda x: (pd.Timestamp(snapshot_date) - x.max()).days),
    Frequency = ('Order_ID',          'count'),
    Monetary  = ('Total_Sales_INR',   'sum')
).reset_index()

# Score each dimension 1–4
rfm['R_Score'] = pd.qcut(rfm['Recency'],   4, labels=[4, 3, 2, 1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'],  4, labels=[1, 2, 3, 4]).astype(int)
rfm['RFM_Score'] = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']

def assign_segment(score):
    if score >= 10: return 'Champions'
    elif score >= 8: return 'Loyal Customers'
    elif score >= 6: return 'Potential Loyalists'
    elif score >= 4: return 'At Risk'
    else:            return 'Lost'

rfm['Segment'] = rfm['RFM_Score'].apply(assign_segment)

# Summary table
seg_summary = rfm.groupby('Segment').agg(
    Customers     = ('Customer_ID', 'count'),
    Avg_Monetary  = ('Monetary',    'mean'),
    Avg_Recency   = ('Recency',     'mean'),
    Avg_Frequency = ('Frequency',   'mean')
).sort_values('Avg_Monetary', ascending=False).round(1)

print("RFM Segment Summary:")
print(seg_summary)
print(f"\nTotal Customers Segmented: {len(rfm):,}")

# Bar chart — customer count per segment
segment_order = ['Champions', 'Loyal Customers', 'Potential Loyalists', 'At Risk', 'Lost']
seg_counts = rfm['Segment'].value_counts().reindex(segment_order)

colors = ['#2ecc71', '#3498db', '#f39c12', '#e67e22', '#e74c3c']
plt.figure(figsize=(10, 5))
ax = sns.barplot(x=seg_counts.index, y=seg_counts.values, palette=colors)
for i, v in enumerate(seg_counts.values):
    ax.text(i, v + 20, f'{v:,}', ha='center', fontsize=11, fontweight='bold')
plt.title('RFM Customer Segmentation — Count by Segment', fontsize=14)
plt.xlabel('Customer Segment')
plt.ylabel('Number of Customers')
plt.tight_layout()
plt.show()

### INSIGHTS ---
RFM segmentation across **6,831 customers** reveals a healthy but skewed distribution: **Champions (1,809 customers, 26.5%)** lead with an average spend of **₹2,75,246** and an average recency of just 61 days, making them the most recently active and highest-value cohort.

**At Risk customers (1,454)** average only **₹58,548 in spend** and haven't purchased in ~227 days — they were once buyers but are drifting away. The **Lost segment (320 customers)** shows the lowest average spend (₹25,374) and a recency of ~291 days, meaning they haven't ordered in nearly 10 months.

**Potential Loyalists (1,734)** are the largest growth opportunity — they have moderate recency and spend (~₹97,048 avg) but low frequency (1.3 orders avg), meaning one well-timed campaign could convert them into Loyal Customers.

**Action:** Build three distinct CRM flows — (1) VIP perks and early access for Champions to maximise retention, (2) a win-back discount campaign for At Risk customers before they cross into Lost, and (3) a "next purchase" nudge with personalised recommendations for Potential Loyalists within 30 days of their last order.

# 🔍 **Key Insights**

- **Festive Demand Peak:**
  Revenue peaks in **August (₹72.2M)** and **July (₹71.5M)** rather than a single Diwali spike — the festive window is broader than expected. **February is the weakest month at ₹60.8M**, about 16% below the August high, marking the clearest trough for clearance and re-engagement campaigns.

- **Payment Method Distribution:**
  All four payment methods are near-equally split — Credit Card (2,776 orders), COD (2,771), UPI (2,683), and Debit Card (2,671) — with AOV virtually identical across all (range: ₹72,901–₹75,126). Payment choice reflects personal preference, not spending power.

- **Order Quantity Behaviour:**
  Median order quantity is **3 units** (mean: 2.99), with a hard cap at 5 units. The tight distribution means bulk-order analysis is limited in this dataset; orders at the 5-unit ceiling are the closest proxy for high-volume buyers.

- **Category Revenue Balance:**
  All 5 categories — **Electronics (₹172.4M), Books (₹171.8M), Beauty (₹171.3M), Clothing (₹167.8M), Home & Kitchen (₹165.3M)** — contribute almost equally (~20% each). Top 2 categories represent 40.6% of revenue with no single dominant category.

- **Delivery & Satisfaction:**
  Delivery status splits evenly across Delivered (33.6%), Returned (33.2%), and Pending (33.2%) — the **33% return rate is a critical operational red flag**. Review ratings are near-identical across all three statuses (~3.0), suggesting ratings are submitted pre-delivery rather than post-receipt.

- **Pricing & Ratings:**
  Near-zero correlation between unit price and review rating (r ≈ 0.00) confirms that customers judge by category expectations, not absolute price. Even ₹50,000 products receive 1-star reviews; even ₹200 products receive 5-star reviews.

- **RFM Customer Segmentation:**
  RFM analysis across **6,831 customers** reveals **Champions (1,809, 26.5%)** averaging ₹2,75,246 in spend vs **Lost customers (320)** at only ₹25,374 — an 11× gap. **Potential Loyalists (1,734)** represent the largest untapped growth cohort with moderate spend but only 1.3 avg orders.

- **Geographic Distribution:**
  State-level revenue is unusually flat — top 3 states (Tripura, Rajasthan, UP) account for only **11.8% of total revenue** — unlike real-world patterns where 3–4 metros typically drive 40–50% of national e-commerce sales.

# 💡 **Recommendations**

1. **Festive Campaign Timing:** Launch inventory build-up, ad campaigns, and warehouse staffing at least 6 weeks before Diwali (mid-September) to capture early-bird and deal-hunter segments, and avoid last-mile congestion in peak week.

2. **Payment Incentives:** Offer UPI/Credit Card-exclusive cashback (e.g., 5–8% off) to nudge buyers toward prepaid rails; for COD customers, promote "Convert to Prepaid" with a ₹100 coupon — reducing return rates and processing cost.

3. **Category & Product Focus:** Concentrate sponsored-ad spend on top-2 revenue categories; cross-sell mid-tier categories as "frequently bought together" add-ons; give high-rated low-revenue products a sponsored visibility boost.

4. **Geographic Expansion:** Use choropleth findings to prioritise next-wave markets — partner with regional logistics players in mid-tier states; run state-specific Diwali promotions in regional languages.

5. **Delivery SLA Improvement:** Map cancelled/pending orders by state and product category; set up real-time alert thresholds for cancellation spikes; proactively notify affected customers with alternatives or compensation.

6. **AOV Uplift via Spend Thresholds:** Design tiered promotional triggers — "Spend ₹5,000 = free delivery / Spend ₹10,000 = 10% cashback" — to nudge mid-tier buyers into the High revenue band.

7. **Customer Retention Programme:** Build a VIP tier for top-10% revenue customers (early deal access, dedicated support, exclusive bundles); run a post-Diwali re-engagement campaign (email/push) for bottom-cohort one-time buyers with personalised recommendations.

8. **Product Quality Action:** Audit all SKUs with Unit Price > ₹5,000 AND Review Rating < 3.5 — these are brand reputation risks; work with sellers to address listing accuracy, packaging, and quality issues before the next campaign.

9. **B2B / Corporate Gifting Channel:** Formalise bulk-order handling with a dedicated corporate gifting portal, volume pricing tiers, and branded packaging to monetise the high-quantity outlier segment as a structured revenue stream.

10. **Dashboard Preparation:** Build Power BI / Tableau aggregated tables for: monthly revenue trend, category performance, state-level choropleth, customer AOV segmentation, and delivery fulfilment KPIs for real-time campaign monitoring.